In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import warnings
warnings.filterwarnings('ignore')

In [ ]:
train_path_input = './data'
test_file_path = '../input/house-prices-advanced-regression-techniques'
#test_file_path = train_path_input # eoh
path_output = '/kaggle/working'
#path_output = './working' # eoh

In [ ]:
# Define constants for attribute names.
columns = ( # Copied from the input file (./data/train.csv)
  'PassengerId','Survived','Pclass','Name','Sex','Age','SibSp','Parch','Ticket','Fare','Cabin','Embarked'
)
    
# Upper case constants for variables (e.g. C.ID, C._1STFLRSF, C._3SSNPORCH, C.SALEPRICE).
class C:
    @classmethod
    def create(cls, names):
        for name in names:
            const_name = name.upper()
            if name[0].isdigit():
                const_name = '_' + const_name
            setattr(cls, const_name, name)
C.create(columns)
test_columns = [columns[0],*columns[2:]]

In [ ]:
# Read in the data.
train_df = pd.read_csv(
    '/'.join([train_path_input,'train.csv']),          # Path to input file.
    header = 0,                                        # Header in line 0.
    names = columns,                                   # Column labels defined.
    index_col = C.PASSENGERID,                         # ID column used for index.
    usecols = columns,                                 # Specify colomns to import.
    skipinitialspace=True,                             # Skip space around ends.
    encoding = "ascii",                                # Assume ascii encoding for input data.
)
test_df = pd.read_csv(
    '/'.join([test_file_path, 'test.csv']),            # Path to input file.
    header = 0,                                        # Header in line 0.
    names = test_columns,                              # Column labels defined.
    index_col = C.PASSENGERID,                         # ID column used for index.
    usecols = test_columns,                            # Specify colomns to import.
    skipinitialspace=True,                             # Skip space around ends.
    encoding = "ascii",                                # Assume ascii encoding for input data.
)

In [ ]:
print("Dataset shape:", train_df.shape)
print("\nFirst few rows:")
print(train_df.head())


# =============================================================================
# DATA EXPLORATION AND PREPROCESSING
# =============================================================================

In [ ]:
def explore_data(df):
    """Explore the dataset structure and missing values"""
    print("Dataset Info:")
    print(df.info())
    print("\nMissing values:")
    print(df.isnull().sum())
    print("\nSurvival rate:")
    print(df['Survived'].value_counts(normalize=True))

explore_data(train_df)

In [ ]:
def preprocess_data(train_df, test_df):
    """Clean and preprocess the data for neural network"""

    # Combine datasets for consistent preprocessing
    full_data = pd.concat([train_df, test_df], ignore_index=False)
  
    # Select comprehensive feature set
    categorical_features = [
      *[C.SEX, C.EMBARKED],  # from columns
      *['Title', 'AgeGroup', 'FareGroup'], # engineered below
    ]

    features = [
      *[C.PCLASS, C.AGE, C.SIBSP, C.PARCH, C.FARE], # from columns
      *['FamilySize', 'IsAlone'], # engineered below
      *[f'{feature}_encoded' for feature in categorical_features], # categorial features to be encoded
    ]

    C.create([*features, *categorical_features])

    full_data = full_data.assign(**{
      # Fill missing values more intelligently
      # Age: Fill with median based on Pclass and Sex
      C.AGE: lambda x: x[C.AGE].fillna(x.groupby([C.PCLASS, C.SEX])[C.AGE].transform('median')),
      # Embarked: Fill with most common value
      C.EMBARKED: lambda x: x[C.EMBARKED].fillna(x[C.EMBARKED].mode()[0]),
      # Fare: Fill with median based on Pclass
      C.FARE: lambda x: x[C.FARE].fillna(x.groupby([C.PCLASS])[C.FARE].transform('median')),

      # Feature engineering
      C.FAMILYSIZE: lambda x: x[C.SIBSP] + x[C.PARCH] + 1,
      C.ISALONE: lambda x: (x[C.FAMILYSIZE] == 1).astype(int),
        
      # Extract title from name
      C.TITLE: lambda x: x[C.NAME].str.extract(
        ' ([A-Za-z]+)\.', 
        expand=False
      ).replace(
        ['Lady','Countess','Capt','Col','Don','Dr','Major','Rev','Sir','Jonkheer','Dona'],
        'Other'
      ).replace('Mlle', 'Miss').replace('Ms', 'Miss').replace('Mme', 'Mrs'),
    
      # Age groups
      C.AGEGROUP: lambda x: pd.cut(
        x[C.AGE], 
        bins=[0, 12, 18, 35, 60, 100],
        labels=['Child', 'Teen', 'Adult', 'Middle', 'Senior']
      ),

      # Fare groups
      C.FAREGROUP: lambda x: pd.qcut(
        x[C.FARE],
        q=4, 
        labels=['Low', 'Medium', 'High', 'VeryHigh']
      ),
    },

    # Encode categorical variables
    **{
      f'{feature}_encoded': lambda x,y=feature:
        LabelEncoder().fit_transform(x[y]) for feature in categorical_features
    }) 

    # Split back to train and test
    train_processed = full_data[:len(train_df)].copy()
    test_processed = full_data[len(train_df):].copy()
    
    return train_processed, test_processed, features

train_processed, test_processed, features = preprocess_data(train_df, test_df)


# =============================================================================
# PREPARE DATA FOR NEURAL NETWORK
# =============================================================================

In [ ]:
# Prepare training data
X = train_processed[features].values
y = train_processed[C.SURVIVED].values

# Split into train and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


In [ ]:
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

print(f"Training set shape: {X_train_scaled.shape}")
print(f"Validation set shape: {X_val_scaled.shape}")
print(f"Number of features: {len(features)}")


# =============================================================================
# BUILD NEURAL NETWORK MODELS
# =============================================================================

In [ ]:
def create_basic_model(input_dim):
    """Create a basic neural network"""
    model = keras.Sequential([
        layers.Dense(64, activation='relu', input_shape=(input_dim,)),
        layers.Dropout(0.3),
        layers.Dense(32, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(16, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])
    
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    
    return model

def create_advanced_model(input_dim):
    """Create a more sophisticated neural network"""
    model = keras.Sequential([
        layers.Dense(128, activation='relu', input_shape=(input_dim,)),
        layers.BatchNormalization(),
        layers.Dropout(0.4),
        
        layers.Dense(64, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        
        layers.Dense(32, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.2),
        
        layers.Dense(16, activation='relu'),
        layers.Dropout(0.1),
        
        layers.Dense(1, activation='sigmoid')
    ])
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy', 'precision', 'recall']
    )
    
    return model

# =============================================================================
# TRAIN MODELS
# =============================================================================

In [ ]:
# Train basic model
print("Training Basic Neural Network...")
basic_model = create_basic_model(X_train_scaled.shape[1])

# Callbacks for training
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=20,
    restore_best_weights=True
)

reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=10,
    min_lr=0.0001
)

In [ ]:
# Train the model
history_basic = basic_model.fit(
    X_train_scaled, y_train,
    epochs=100,
    batch_size=32,
    validation_data=(X_val_scaled, y_val),
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

In [ ]:
# Train advanced model
print("\nTraining Advanced Neural Network...")
advanced_model = create_advanced_model(X_train_scaled.shape[1])

history_advanced = advanced_model.fit(
    X_train_scaled, y_train,
    epochs=100,
    batch_size=32,
    validation_data=(X_val_scaled, y_val),
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

# =============================================================================
# EVALUATE MODELS
# =============================================================================

In [ ]:
def evaluate_model(model, X_val, y_val, model_name):
    """Evaluate model performance"""
    print(f"\n{model_name} Evaluation:")
    
    # Predictions
    val_pred_prob = model.predict(X_val)
    val_pred = (val_pred_prob > 0.5).astype(int)
    
    # Accuracy
    accuracy = accuracy_score(y_val, val_pred)
    print(f"Validation Accuracy: {accuracy:.4f}")
    
    # Classification report
    print("\nClassification Report:")
    print(classification_report(y_val, val_pred))
    
    # Confusion matrix
    cm = confusion_matrix(y_val, val_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'{model_name} - Confusion Matrix')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.show()
    
    return accuracy

# Evaluate models
basic_accuracy = evaluate_model(basic_model, X_val_scaled, y_val, "Basic Neural Network")
advanced_accuracy = evaluate_model(advanced_model, X_val_scaled, y_val, "Advanced Neural Network")

# =============================================================================
# PLOT TRAINING HISTORY
# =============================================================================

In [ ]:
def plot_training_history(history, model_name):
    """Plot training and validation metrics"""
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Plot accuracy
    axes[0].plot(history.history['accuracy'], label='Training Accuracy')
    axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy')
    axes[0].set_title(f'{model_name} - Accuracy')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(True)
    
    # Plot loss
    axes[1].plot(history.history['loss'], label='Training Loss')
    axes[1].plot(history.history['val_loss'], label='Validation Loss')
    axes[1].set_title(f'{model_name} - Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True)
    
    plt.tight_layout()
    plt.show()

plot_training_history(history_basic, "Basic Neural Network")
plot_training_history(history_advanced, "Advanced Neural Network")


# =============================================================================
# FEATURE IMPORTANCE ANALYSIS
# =============================================================================

In [ ]:
def analyze_feature_importance(model, feature_names, X_val):
    """Analyze feature importance using permutation"""
    baseline_accuracy = accuracy_score(y_val, (model.predict(X_val) > 0.5).astype(int))
    
    feature_importance = []
    
    for i, feature in enumerate(feature_names):
        # Create a copy of validation data
        X_val_permuted = X_val.copy()
        
        # Permute the feature
        X_val_permuted[:, i] = np.random.permutation(X_val_permuted[:, i])
        
        # Calculate accuracy with permuted feature
        permuted_accuracy = accuracy_score(y_val, (model.predict(X_val_permuted) > 0.5).astype(int))
        
        # Importance is the drop in accuracy
        importance = baseline_accuracy - permuted_accuracy
        feature_importance.append((feature, importance))
    
    # Sort by importance
    feature_importance.sort(key=lambda x: x[1], reverse=True)
    
    # Plot feature importance
    features_sorted, importance_sorted = zip(*feature_importance)
    
    plt.figure(figsize=(12, 8))
    plt.barh(range(len(features_sorted)), importance_sorted)
    plt.yticks(range(len(features_sorted)), features_sorted)
    plt.xlabel('Importance (Drop in Accuracy)')
    plt.title('Feature Importance Analysis')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
    
    return feature_importance

print("\nAnalyzing Feature Importance...")
feature_importance = analyze_feature_importance(advanced_model, features, X_val_scaled)

print("\nTop 5 Most Important Features:")
for feature, importance in feature_importance[:5]:
    print(f"{feature}: {importance:.4f}")

# =============================================================================
# PREPARE SUBMISSION
# =============================================================================

In [ ]:
def prepare_submission(model, test_processed, features, scaler):
    """Prepare submission file for Kaggle"""
    
    # Prepare test data
    X_test = test_processed[features].values
    X_test_scaled = scaler.transform(X_test)
    
    # Make predictions
    test_predictions = model.predict(X_test_scaled)
    test_predictions = (test_predictions > 0.5).astype(int).flatten()
    
    # Create submission dataframe
    submission = pd.DataFrame(
      data = test_predictions,
      columns = [C.SURVIVED],
      index = test_processed.index
    )
    
    # Save to 
    output_file = 'titanic_neural_network_submission.csv'
    path = '/'.join([path_output, output_file])
    submission.to_csv(path, index=True)
    print(f"Submission file saved as {path}")
    
    return submission

# Create submission with the best model
if advanced_accuracy > basic_accuracy:
    submission = prepare_submission(advanced_model, test_processed, features, scaler)
    print(f"Using Advanced Neural Network (Accuracy: {advanced_accuracy:.4f})")
else:
    submission = prepare_submission(basic_model, test_processed, features, scaler)
    print(f"Using Basic Neural Network (Accuracy: {basic_accuracy:.4f})")

print("\nSubmission Preview:")
print(submission.head(10))

# =============================================================================
# SUMMARY AND TIPS
# =============================================================================

In [ ]:
print("\n" + "="*60)
print("NEURAL NETWORK MODELING SUMMARY")
print("="*60)
print(f"Basic Neural Network Accuracy: {basic_accuracy:.4f}")
print(f"Advanced Neural Network Accuracy: {advanced_accuracy:.4f}")
print(f"Total Features Used: {len(features)}")
print(f"Training Set Size: {len(X_train)}")
print(f"Validation Set Size: {len(X_val)}")

print("\nKey Techniques Used:")
print("1. Feature Engineering (FamilySize, IsAlone, Title extraction)")
print("2. Data Preprocessing (Missing value imputation, encoding)")
print("3. Feature Scaling (StandardScaler)")
print("4. Dropout Layers (Regularization)")
print("5. Batch Normalization (Advanced model)")
print("6. Early Stopping (Prevent overfitting)")
print("7. Learning Rate Reduction")
print("8. Feature Importance Analysis")

print("\nTips for Improvement:")
print("1. Try ensemble methods (combine multiple models)")
print("2. Hyperparameter tuning with Keras Tuner")
print("3. Cross-validation for more robust evaluation")
print("4. Additional feature engineering")
print("5. Try different architectures (wider/deeper networks)")
print("6. Experiment with different activation functions")